# Reflexion (fiel a Shinn et al. 2023)

**Ref.:** Shinn et al. (2023), *Reflexion: Language Agents with Verbal Reinforcement Learning*, arXiv:2303.11366.

Tres componentes:
- **Actor:** recomienda condicionado por su **memoria episodica verbal** (buffer de reflexiones).
- **Evaluator:** senal binaria exito/fallo = hit@1 (soft-match coseno vs GT).
- **Self-Reflection:** SOLO ante un fallo, escribe una reflexion verbal y la **agrega al buffer**.

**Adaptacion:** acumulamos la memoria verbal **entre dialogos** (warmup sobre train) y la
**congelamos** para evaluar en test, por comparabilidad con ACE-offline. Las reflexiones se
piden **generales** (no nombrar el target) para que transfieran y no memoricen respuestas.

**Consistencia con el proyecto:** datos y metricas se importan de `utils` (mismo `build_context`,
mismo soft-matching: Recall@1/@5, MRR, Novelty, BERTScore; threshold=0.90, gt_field=`gt_accepted`).

## 0. Dependencias

In [ ]:
# === Dependencias (correr una vez) ===
!pip install -q transformers accelerate bitsandbytes sentence-transformers rapidfuzz evaluate bert_score
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
GPU: Tesla T4


## 1. Montar Drive y ubicarse en la carpeta del proyecto

In [ ]:
# === Drive: montar y posicionarse en la carpeta del proyecto ===============
# La carpeta esta en "Compartidos conmigo", que Colab NO monta directamente.
# Una sola vez en la web de Drive: click derecho sobre "Proyecto RecSys" ->
# Organizar -> Anadir acceso directo -> "Mi unidad" (es solo un puntero).
import os, sys
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys"
assert os.path.isdir(PROJECT_DIR), (
    "No existe " + PROJECT_DIR + ". Crea el acceso directo en Mi unidad.")
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
PATH = PROJECT_DIR + "/"
print("Trabajando en:", os.getcwd())
print("Archivos:", [f for f in os.listdir() if f.startswith("utils")])

Mounted at /content/drive
Trabajando en: /content/drive/.shortcut-targets-by-id/1DVeLSUjM_ZgsdoEI3aeO7kyHJTdoFuLR/Proyecto RecSys
Archivos: ['utils.ipynb', 'utils.py']


## 2. utils + datos (sin fuga)

In [ ]:
# === Bootstrap de utils + carga de datos ===================================
import os, json

if not os.path.exists("utils.py"):
    if os.path.exists("utils.ipynb"):
        _nb = json.load(open("utils.ipynb", encoding="utf-8"))
        with open("utils.py", "w", encoding="utf-8") as f:
            for c in _nb["cells"]:
                if c["cell_type"] == "code":
                    f.write("".join(c["source"]) + "\n\n")
        print("utils.py generado desde utils.ipynb")
    else:
        raise FileNotFoundError("No encuentro utils.ipynb ni utils.py en " + os.getcwd())

from utils import (
    download_dataset, load_parsed, sample_conversations,
    build_context, build_train_freq, build_recommender_references,
    run_full_evaluation, soft_hit,
)

DATASET = "pearl"            # cambiar a "pearl" para correr en PEARL
paths = download_dataset(DATASET, splits=("train", "test"))
train_parsed = load_parsed(paths["train"], dataset=DATASET)
test_parsed  = load_parsed(paths["test"], dataset=DATASET)
print(f"Train: {len(train_parsed)} | Test: {len(test_parsed)}")

# --- Sampling reproducible (mismo seed para todos los metodos) ---
N_WARMUP = 100
N_EVAL   = 300
GT_FIELD = "gt_accepted"
warmup_sample = sample_conversations(train_parsed, n=N_WARMUP, seed=42, gt_field=GT_FIELD)
eval_sample   = sample_conversations(test_parsed,  n=N_EVAL,   seed=42, gt_field=GT_FIELD)
print(f"Warmup: {len(warmup_sample)} | Eval: {len(eval_sample)}")

pearl_train.json ya existe, omitiendo descarga.
pearl_test.json ya existe, omitiendo descarga.
Train: 50000 | Test: 2277
Sampled 100 conversaciones con gt_accepted no vacío (seed=42)
Sampled 300 conversaciones con gt_accepted no vacío (seed=42)
Warmup: 100 | Eval: 300


## 3. Modelo base — Qwen3-8B 4-bit

In [ ]:
# === Modelo base: Qwen3-8B 4-bit (cabe en T4) ==============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, json, re

MODEL_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto")

def call_llm(messages, max_tokens=220):
    """Inferencia determinista (greedy, sin bloque <think>)."""
    text = tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

def parse_json(raw):
    """Extrae el primer objeto JSON de la salida del modelo."""
    try:
        return json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group())
    except Exception:
        return {}

def dedup_titles(titles):
    """Quita titulos duplicados preservando orden (normaliza sin anio)."""
    seen, out = set(), []
    for t in titles:
        k = t.split(" (")[0].strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(t)
    return out


print("Modelo cargado.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado.


## 4. Artefactos de métricas

In [ ]:
# === Embeddings + artefactos para Novelty/BERTScore + reward del Evaluator ==
from sentence_transformers import SentenceTransformer

device_embed = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)

train_freq, n_train = build_train_freq(paths["train"], dataset=DATASET)
human_refs = build_recommender_references(paths["test"], dataset=DATASET)

THRESHOLD = 0.90   # identico al de las metricas (utils)

def gt_titles_of(d, field=GT_FIELD):
    ts = [d["title_map"].get(m, "").split(" (")[0].strip() for m in d[field]]
    return [t for t in ts if t]

# soft_hit se importa de utils (mismo criterio que evaluate_soft); recibe embed_model.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Train freq construido: 9303 títulos únicos en 50000 diálogos
Referencias construidas: 2277 diálogos


## 5. Salida (Drive)

In [ ]:
# Resultados se guardan en PATH (definido en la celda de Drive).
print("Resultados ->", PATH)

Resultados -> /content/drive/MyDrive/Proyecto RecSys/


## 6. Componentes Reflexion

In [ ]:
# === Componentes Reflexion =================================================
import time
MAX_REFLECTIONS = 15   # tamano del buffer episodico (ventana deslizante)

# --- Actor (prompt recomendador acordado por el grupo) ---
# message: explica la 1a recomendacion con un motivo y menciona las otras 4.
def actor_recommend(dialogue, buffer, max_tokens=320):
    mem = ""
    if buffer:
        mem = ("Lessons you learned from previous recommendation attempts "
               "(apply them):\n" + "\n".join(f"- {r}" for r in buffer) + "\n\n")
    system = (
        mem +
        "You are a conversational movie recommender.\n\n"
        "CRITICAL RULE: Under NO circumstances may you recommend a movie that "
        "has already been mentioned by ANYONE (User or Assistant) anywhere in "
        "the conversation history. Every one of your 5 recommendations must be "
        "a movie NOT YET discussed in this conversation — genuinely new "
        "suggestions the user hasn't heard yet.\n\n"
        "Recommend exactly 5 movies, best match first. Include the release "
        "year formatted exactly as 'Title (Year)'.\n"
        "Respond ONLY with valid JSON:\n"
        '{"message": "I recommend [Title (Year)] because [brief reason]. '
        'You might also enjoy [T2], [T3], [T4], and [T5].", '
        '"structured_recommendations": ["Title (Year)", "T2", "T3", "T4", "T5"]}\n'
        "No text outside the JSON."
    )
    raw = call_llm([{"role": "system", "content": system},
                    {"role": "user", "content": build_context(dialogue)}], max_tokens=max_tokens)
    data = parse_json(raw)
    recs = dedup_titles(data.get("structured_recommendations", []) or [])
    return recs, data.get("message", ""), raw

# --- Evaluator (binario, soft-match en la 1a prediccion) ---
def evaluator(movies, gt_titles):
    return bool(movies) and soft_hit(movies[0], gt_titles, EMBED_MODEL, THRESHOLD)

# --- Self-Reflection (solo en fallo; lecciones GENERALES, transferibles) ---
def self_reflect(context, movies, gt_titles, max_tokens=160):
    system = (
        "You are the SELF-REFLECTION module of a movie recommender that just FAILED.\n"
        "Write a concise verbal lesson (2-3 sentences) diagnosing WHY the recommendation "
        "missed and a GENERAL strategy to do better next time.\n"
        "Do NOT name the human target titles; keep the lesson transferable to other users."
    )
    user = (f"User request:\n{context}\n\nYou recommended: {movies}\n"
            f"The human target was: {gt_titles}\n\nWrite the reflection:")
    return call_llm([{"role": "system", "content": system},
                     {"role": "user", "content": user}], max_tokens=max_tokens).strip()

## 7. Warmup — construir el buffer (dirigido por error)

In [ ]:
# === Warmup sobre train: buffer de reflexiones, dirigido por error =========
def reflexion_warmup(sample, save_path=None):
    buffer, hits, miss = [], 0, 0
    t0 = time.time()
    for i, d in enumerate(sample):
        gts = gt_titles_of(d)
        if not gts:
            continue
        movies, _, _ = actor_recommend(d, buffer)
        ok = evaluator(movies, gts)
        hits += int(ok); miss += int(not ok)
        if not ok:
            r = self_reflect(build_context(d), movies[:3], gts[:3])
            if r:
                buffer.append(r)
                buffer = buffer[-MAX_REFLECTIONS:]   # ventana deslizante
        if (i + 1) % 25 == 0:
            print(f"[{i+1}/{len(sample)}] hit@1={hits/max(hits+miss,1):.3f} | "
                  f"buffer={len(buffer)} | ETA {((time.time()-t0)/(hits+miss))*(len(sample)-i-1)/60:.1f} min")
            if save_path: json.dump(buffer, open(save_path, "w"), indent=2)
    print(f"\nWarmup listo | buffer final={len(buffer)} reflexiones")
    return buffer

_frozen = PATH + f"reflexion_buffer_frozen_{DATASET}.json"
if os.path.exists(_frozen):
    reflection_buffer = json.load(open(_frozen))
    print(f"Buffer congelado cargado ({len(reflection_buffer)} reflexiones) — se omite el warmup.")
else:
    reflection_buffer = reflexion_warmup(warmup_sample, save_path=PATH + f"reflexion_buffer_{DATASET}.json")
    json.dump(reflection_buffer, open(_frozen, "w"), indent=2)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[25/100] hit@1=0.000 | buffer=15 | ETA 31.1 min
[50/100] hit@1=0.000 | buffer=15 | ETA 20.5 min
[75/100] hit@1=0.000 | buffer=15 | ETA 10.3 min
[100/100] hit@1=0.000 | buffer=15 | ETA 0.0 min

Warmup listo | buffer final=15 reflexiones


## 8. Evaluación (buffer congelado)

In [ ]:
# === Evaluacion en test con buffer congelado ===============================
# idx = original_idx para alinear con test_parsed (GT) y human_refs (BERTScore).
def reflexion_eval(sample, buffer, out_path, save_every=25):
    outs, done = [], set()
    if os.path.exists(out_path):
        outs = json.load(open(out_path)); done = {o["sample_pos"] for o in outs}
    t0 = time.time()
    for pos, d in enumerate(sample):
        if pos in done:
            continue
        movies, msg, raw = actor_recommend(d, buffer)
        outs.append({"sample_pos": pos, "idx": d["original_idx"],
                     "predicted": movies, "message": msg, "raw": raw})
        n = len(outs)
        if n % save_every == 0:
            json.dump(outs, open(out_path, "w"))
            eta = ((time.time()-t0)/max(n-len(done),1))*(len(sample)-n)/60
            print(f"{n}/{len(sample)} | ETA {eta:.1f} min")
    json.dump(outs, open(out_path, "w"))
    return outs

out_ref = reflexion_eval(eval_sample, reflection_buffer, PATH + f"reflexion_results_{DATASET}.json")

print("\n=== Reflexion — metricas (soft-matching) ===")
metrics_ref = run_full_evaluation(
    out_ref, test_parsed, EMBED_MODEL,
    train_freq, n_train, human_refs,
    threshold=THRESHOLD, gt_field=GT_FIELD, k_list=(1, 5),
)
json.dump(metrics_ref, open(PATH + f"reflexion_metrics_{DATASET}.json", "w"), indent=2)
print(metrics_ref)

25/300 | ETA 81.5 min
50/300 | ETA 73.2 min
75/300 | ETA 65.7 min
100/300 | ETA 58.1 min
125/300 | ETA 50.9 min
150/300 | ETA 43.5 min
175/300 | ETA 36.3 min
200/300 | ETA 29.0 min
225/300 | ETA 21.8 min
250/300 | ETA 14.6 min
275/300 | ETA 7.3 min
300/300 | ETA 0.0 min

=== Reflexion — metricas (soft-matching) ===
  EVALUACIÓN COMPLETA

── Recall & MRR (soft-matching) ──
Evaluable: 300 / 300
  Recall@1: 0.0100
  Recall@5: 0.0233
  MRR:      0.0143

── Novelty ──
  Novelty:  10.77 bits (norm: 0.690)
  No vistas en train: 19.6% (284/1451)

── BERTScore ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERTScore (294/300): P=0.8705  R=0.8834  F1=0.8768

{'Recall@1': 0.01, 'Recall@5': 0.0233, 'MRR': 0.0143, 'n_evaluable': 294, 'novelty_mean': 10.7718, 'novelty_norm': 0.6901, 'unseen_rate': 0.1957, 'n_dialogos': 294, 'n_recs': 1451, 'precision': 0.8705, 'recall': 0.8834, 'f1': 0.8768, 'n_total': 300}
